Imports

In [64]:
# Calculations
import calfem.core as cfc
import numpy as np
import calfem.utils as cfu
import math
from scipy.integrate import solve_ivp
from plantml import plantml

# Mesh
import calfem.geometry as cfg
import calfem.mesh as cfm

# Visual
import calfem.vis_mpl as cfv
import matplotlib as mpl
mpl.use('TkAgg')

<h1>Constants for Project</h1>

In [65]:
E = 5e9 # Young's modulus [Pa]
nu = 0.36 # Poisson's ratio [-]
alpha_exp = 60e-6 # Expansion coefficient [1/K]
rho = 549 # Density [kg/m^3]
c_p = 3600 # Specific heat capacity [J/(kg*K)]
k = 80 # Thermal conductivity [W/(m*K)]
alpha_n = 40 # Convection coefficient for top boundary [W/(m^2*K)]
alpha_c = 120 # Convection coefficient for cooling channels [W/(m^2*K)]
T_inf = 293 # Temperature of surrounding environment as well as starting temperature T_0 [W/(m^2*K)]
T_out = 285 # Temperature of hot cooling channels [W/(m^2*K)]
T_in = 277 # Temperature of cold cooling channels [W/(m^2*K)]
thickness = 1.6 # Thickness [m]

el_size_factor = 0.03 # Size factor of the elements [-]

<h2>Pre-processing</h2>

Setting up geometry

In [66]:
# Boundary markers
MARKER_T_ENV = 1                # For splines where convection occurs with the surrounding environment (boundary condition)
MARKER_T_OUT = 2                # For splines where convection occurs with the hot cooling channels (boundary condition)
MARKER_T_IN = 3                 # For splines where convection occurs with the cold cooling channels (boundary condition)
MARKER_U_DISPLACEMENT = 5       # For splines where the displacement is 0 (boundary condition)
MARKER_U_DISPLACEMENT_X =  6    # For splines where the displacement on the x-axis is 0 (symmetry condition)


g = cfg.Geometry()
#!!!### ha med i rapporten motsolsplacering

# POINTS
# corner                                            # INDEX
g.point([0.0, 0.0])                                 # 0       
g.point([0.300, 0.0])                               # 1
g.point([0.400, 0.200])                             # 2
g.point([0.0, 0.200])                               # 3

# half cooling channel
g.point([0.0, 0.100])                               # 4
g.point([0.0, 0.125])                               # 5
g.point([0.025, 0.100])                             # 6
g.point([0.0, 0.075])                               # 7

# CURVES
                                                    # INDEX
g.spline([0, 1], marker=MARKER_U_DISPLACEMENT)      # 0 (bottom)
g.spline([1, 2], marker=MARKER_U_DISPLACEMENT)      # 1 (diagonal edge right)
g.spline([2, 3], marker=MARKER_T_ENV)               # 2 (top)
g.spline([3, 5], marker=MARKER_U_DISPLACEMENT_X)    # 3 (left top straight edge)
g.circle([5, 4, 6], marker=MARKER_T_OUT)            # 4 (top quarter circle of half cooling channel)
g.circle([6, 4, 7], marker=MARKER_T_OUT)            # 5 (bottom quarter circle of half cooling channel)
g.spline([7, 0], marker=MARKER_U_DISPLACEMENT_X)    # 6 (left bottom straight edge)

# HOLES
def create_hole(center, radius, point_startid, curve_startid, marker):
    g.point(center)
    g.point([center[0], center[1] + radius])
    g.point([center[0] + radius, center[1]])
    g.point([center[0], center[1] - radius])
    g.point([center[0] - radius, center[1]])
    g.circle([point_startid + 1, point_startid, point_startid + 2], marker=marker)
    g.circle([point_startid + 2, point_startid, point_startid + 3], marker=marker)
    g.circle([point_startid + 3, point_startid, point_startid + 4], marker=marker)
    g.circle([point_startid + 4, point_startid, point_startid + 1], marker=marker)
    return [curve_startid, curve_startid + 1, curve_startid + 2, curve_startid + 3]

hole1 = create_hole([0.0875, 0.100], 0.025, 8, 7, MARKER_T_IN)
hole2 = create_hole([0.0875 * 2.0, 0.100], 0.025, 13, 11, MARKER_T_OUT)
hole3 = create_hole([0.0875 * 3.0, 0.100], 0.025, 18, 15, MARKER_T_IN)

# Generating surface
g.surface([0, 1, 2, 3, 4, 5, 6], [hole1, hole2, hole3])

# Visualising final geometry
cfv.draw_geometry(g)
cfv.show()

Generating mesh

In [67]:
mesh = cfm.GmshMesh(g)
mesh.el_type = 2                        # Element type: Triangles
mesh.dofs_per_node = 1                  # Temperature <=> 1 degree of freedom per node
mesh.el_size_factor = el_size_factor
mesh.returnBoundaryElements = True      # To later retrieve which elements are subject to convection

coords, edof, dofs, bdofs, elementmarkers, boundaryElements = mesh.create()

cfv.figure()

# Draw the mesh.
cfv.draw_mesh(
    coords=coords,
    edof=edof,
    dofs_per_node=mesh.dofsPerNode,
    el_type=mesh.elType,
    filled=True,
    title="Battery"
        )
cfv.show()

<h2>Solver</h2>

<h3>Stationary Temperature distribution</h3>

FEM Setup

In [68]:
ndof = len(coords)
nelem = len(edof)
[ex, ey] = cfc.coordxtr(edof,coords,dofs,3)
K=np.zeros((ndof, ndof))
f=np.zeros((ndof, 1))

Function to apply convection (add K_c to K and the )XXXXXXXXXX

In [69]:
def convection_boundary(boundaryElements, K, f, T, alpha, marker):
    for bel in boundaryElements[marker]:
        node1 = bel['node-number-list'][0]          # Node number (1-based index)
        node2 = bel['node-number-list'][1]          # Node number (1-based index)
        node1i = node1 - 1                          # Node index (0-based index)
        node2i = node2 - 1                          # Node index (0-based index)

        # The coords vector uses 0-based index, thus node1i and node2i
        L = math.sqrt((coords[node1i][0] - coords[node2i][0])**2 + (coords[node1i][1]-coords[node2i][1])**2)
        Kce = np.array([[2, 1], [1, 2]]) * alpha * L/6 * thickness
        fbc = np.array([[1/2], [1/2]]) * alpha * T * L * thickness

        # The assem function uses edof (which we directly input as the retrieved nodes). Edof is 1-based, thus node1 and node2.
        cfc.assem(np.array([node1, node2]), K, Kce, f, fbc)

Calculating stiffness matrix, adding convection and solving the final equation!

In [70]:
# The material is assumed to be isotropic, thus the consitutive matrix becomes:
D_constitutive = k*np.eye(2)

# Looping through each element, calculating the stiffness contribution using CALFEM and then assembling.
for i in range(nelem):
    ke = cfc.flw2te(ex[i,:], ey[i,:], [thickness], D_constitutive)
    cfc.assem(edof[i,:], K, ke)

# Adding convection
convection_boundary(boundaryElements, K, f, T_inf, alpha_n, MARKER_T_ENV)
convection_boundary(boundaryElements, K, f, T_out, alpha_c, MARKER_T_OUT)
convection_boundary(boundaryElements, K, f, T_in, alpha_c, MARKER_T_IN)

# We have no Dirichlet conditions but the empty arrays are nevertheless needed for cfc.solveq to not throw an exception
bc, bc_value = np.array([], 'i'), np.array([], 'f')

# Solving!
a, r = cfc.solveq(K, f, bc, bc_value)

# Drawing Temperatures!
cfv.figure(fig_size=(10,5))
# Extracting min and max, temperatures as well as max deviation
nodei_max = np.argmax(a)
nodei_min = np.argmin(a)
T_max = a[nodei_max][0, 0]
T_min = a[nodei_min][0, 0]
max_deviation = (T_min - T_inf) if abs(T_min - T_inf) > abs(T_max - T_inf) else (T_max - T_inf)
# Draw right half
cfv.draw_nodal_values(a, coords, edof, dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type)
# Mirror to the left half!
cfv.draw_nodal_values(a, coords * [-1, 1], edof, dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type)
# Print max and min temperatures as well as their positions!
print("T_min = " + str(T_min) + "K at coordinate: " + str(coords[nodei_min]))
print("T_max = " + str(T_max) + "K at coordinate: " + str(coords[nodei_max]))
print("Maximal Deviation = " + str(max_deviation))
# Mark the positions of the max and min temperatures in the plot.
cfv.plt.plot(coords[nodei_min][0], coords[nodei_min][1], marker='X', markeredgecolor="red", markerfacecolor= "red")
cfv.plt.plot(coords[nodei_max][0], coords[nodei_max][1], marker='X', markeredgecolor="purple", markerfacecolor= "purple")
cfv.plt.plot(-coords[nodei_min][0], coords[nodei_min][1], marker='X', markeredgecolor="red", markerfacecolor= "red")
cfv.plt.plot(-coords[nodei_max][0], coords[nodei_max][1], marker='X', markeredgecolor="purple", markerfacecolor= "purple")
# Adding a color bar
cbar = cfv.plt.colorbar()
cbar.set_label("K", loc="center")
# Labels
cfv.title("Stationary temperature distribution")
cfv.plt.xlabel("x [m]")
cfv.plt.ylabel("y [m]")

cfv.show()

T_min = 282.5369668812037K at coordinate: [0.2625 0.075 ]
T_max = 283.5577492442907K at coordinate: [0.4 0.2]
Maximal Deviation = -10.463033118796318


<h3>Transient </h3>

FEM Setup

In [99]:
# The number of seconds that we're studying the battery transient
time_length = 3600
# The number of steps we take in the time integration
time_steps = 100
# The time stamps we want to plot (excluding the one where the deviation is at its largest, which we calculate below)
times_to_plot = [10.0, 300.0, 600.0, 650.0, 3599.0]
# We convert these time points to indices.
indices_to_plot = [int(e * (time_steps / time_length)) for e in times_to_plot]

# Defining given functions where Q_1 was given as f_1, and Q_2 as f_2 [W/m^3]
def Q_1(t):
    return 100*np.e**(-144*((600 - t)/3600)**2) * 1000

def Q_2(t):
    return 88.42*1000 if t <= 600 else 0

# We are reusing the same mesh as generated for the stationary problem.

# Initializing matrices
K=np.zeros((ndof, ndof))
# f contribution from heat supply
fq=np.zeros((ndof, 1))
# f contribution from convection.
fconv = np.zeros((ndof, 1))
C=np.zeros((ndof, ndof))

## Initializing storage of Temperature distributions
# Evenly distributed time points
ts = np.linspace(0, time_len, time_steps)
# A y (Temperature distribion) for each timepoint for f_1
ys_1 = np.empty((time_steps, ndof))
# A y (Temperature distribution) for each timepoint for f_2
ys_2 = np.empty((time_steps, ndof))
# The temperature distribution is the environment temperature T_inf everywhere
ys_1[0, :] = np.full((ndof,), T_inf)
ys_2[0, :] = np.full((ndof,), T_inf)

In [113]:
# Calculating stiffness matrix Ke and the Ce matrix followed by assembling 
for i in range(nelem):
    # This is the same function call as for the stationary case, but we now have a heat supply dictated by f_1 or f_2. See report for why we can input 1 as temperature supply here (space independence).
    ke, fqe = cfc.flw2te(ex[i,:], ey[i,:], [thickness], D_constitutive, 1)
    # The given plantml calculates the integral for Ce given these parameters.
    Ce = plantml(ex[i,:],ey[i,:], rho*c_p*thickness)
    cfc.assem(edof[i,:], K, ke, fq, fqe)
    cfc.assem(edof[i,:], C, Ce)

# Add convection to the stiffness matrix.
convection_boundary(boundaryElements, K, fconv, T_inf, alpha_n, MARKER_T_ENV)
convection_boundary(boundaryElements, K, fconv, T_out, alpha_c, MARKER_T_OUT)
convection_boundary(boundaryElements, K, fconv, T_in, alpha_c, MARKER_T_IN)

# Implicit Euler for solving the transient behaviour.
dt = ts[1] - ts[0]
for t_i in range(1, len(ts)):
    # Adding the f terms and multiplying the fq integral with the Q given by Q_1 for the given timepoint. See report for explanation.
    f_1 = fq * Q_1(ts[t_i]) + fconv
    # Same but with Q_2
    f_2 = fq * Q_2(ts[t_i]) + fconv
    # Here we divide the calculations for y_{n+1} to increase readibility and reduce calculations by only calculating left once.
    left = np.linalg.inv(C + dt*K)
    right_1 = dt*f_1.flatten() + (C@(ys_1[t_i - 1]).T)
    ys_1[t_i] = (left@(right_1)).T
    right_2 = dt*f_2.flatten() + (C@(ys_2[t_i - 1]).T)
    ys_2[t_i] = (left@(right_2)).T

# Defining a function to reduce clutter in code for extracting the sought after min, max and deviation arrays.
def extract_min_max_dev(ys):
    max_ys = np.zeros((time_steps,))
    min_ys = np.zeros((time_steps,))
    deviation = np.zeros((time_steps,))
    for i in range(time_steps):
        max_val = max(ys[i])
        min_val= min(ys[i])
        max_dev = max(abs(max_val - T_inf), abs(min_val - T_inf))
        max_ys[i] = max_val
        min_ys[i] = min_val
        deviation[i] = max_dev
    return min_ys, max_ys, deviation

min_ys_1, max_ys_1, deviation_1 = extract_min_max_dev(ys_1)
min_ys_2, max_ys_2, deviation_2 = extract_min_max_dev(ys_2)


def plot_interesting_time_points(ys, deviations, heat_func_name):
    max_dev_i = np.argmax(deviations)
    indices = indices_to_plot + [max_dev_i]
    indices.sort()
    for i in range(len(indices)):
        cfv.figure()
        cfv.draw_nodal_values(ys[indices[i]], coords, edof, dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type)
        cfv.draw_nodal_values(ys[indices[i]], coords * [-1, 1], edof, dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type)
        # Adding a color bar
        cbar = cfv.plt.colorbar()
        cbar.set_label("K", loc="center")
        # Labels
        cfv.title(f"Temperature distribution at t = {indices[i] * time_length / time_steps} s" + " (max deviatoin)" * (indices[i] == max_dev_i) + f" for {heat_func_name}")
        cfv.plt.xlabel("x [m]")
        cfv.plt.ylabel("y [m]")

plot_interesting_time_points(ys_1, deviation_1, "Q_1")
plot_interesting_time_points(ys_2, deviation_2, "Q_2")


cfv.figure()
cfv.plt.plot(ts, max_ys_1, color = "red")
cfv.plt.plot(ts, max_ys_2, color = "blue")
cfv.title("Maximum temperature in battery over time")
cfv.plt.xlabel("t [s]")
cfv.plt.ylabel("T [K]")
cfv.plt.legend(["f_1", "f_2"])
cfv.figure()
cfv.plt.plot(ts, min_ys_1, color = "red")
cfv.plt.plot(ts, min_ys_2, color = "blue")
cfv.title("Minimum temperature in battery over time")
cfv.plt.xlabel("t [s]")
cfv.plt.ylabel("T [K]")
cfv.plt.legend(["f_1", "f_2"])
cfv.figure()
cfv.plt.plot(ts, deviation_1, color = "red")
cfv.plt.plot(ts, deviation_2, color = "blue")
cfv.title("Maximum deviation in temperature in battery over time (|T-T_0|)")
cfv.plt.xlabel("t [s]")
cfv.plt.ylabel("T [K]")
cfv.plt.legend(["f_1", "f_2"])
cfv.show()

C:\Users\samue\AppData\Local\Temp\ipykernel_18940\1668531459.py:59: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  cfv.title(f"Temperature distribution at t = {indices[i] * time_length / time_steps} s" + " (max deviatoin)" * (indices[i] == max_dev_i) + f" for {heat_func_name}")


KeyboardInterrupt: 

In [ ]:
storlek, timesteps